# TrOCR run 6: augmented mixed fine-tune

Same recipe as run 5 (synthetic print + real EHRI typewritten Polish lines),
but with **5× augmented** EHRI train lines (349 → 2094 lines). run 5 plateaued
at 30 epochs on 349 real lines, so more data — not more epochs — is the next
lever. Augmentations simulate typewriter/scan degradation: blur, noise,
contrast, gamma, JPEG, rotation. 15 epochs (enough per run 4/5 plateau analysis).

Starts fresh from `PiotrSty/trocr-pl-base`. Evaluates run 6 against run 5
(`trocr-pl-mixed-v3`), run 4 (`trocr-pl-mixed-v2`), and run 2 (`trocr-pl-base`)
on the frozen EHRI test docs and the printed `real-lines-v1` benchmark.
Select GPU T4. All revisions pinned.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '410b404233516e38ea94da8d8fc3c4085b777ea0'
BASE_MODEL = 'PiotrSty/trocr-pl-base'
BASE_REVISION = 'fff0416a9ccd8786cbd6f12d4cf3147c07056b18'
RUN3_MODEL = 'PiotrSty/trocr-pl-mixed-v1'
RUN4_MODEL = 'PiotrSty/trocr-pl-mixed-v2'
RUN5_MODEL = 'PiotrSty/trocr-pl-mixed-v3'
SYN_REVISION = 'd881debb90045fd71ad8e25faeeafeb6adab6622'
EHRI_REVISION = '3003e8614b74a351e7d94aba4f1348368815fb70'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)
# Kaggle base image ships huggingface_hub 1.11.0; a plain downgrade leaves stale
# files that break transformers 4.57.6 (ImportError: HfFolder from utils). Fully
# uninstall first, then no-cache reinstall 0.36.2 (compatible with transformers 4.57.6).
subprocess.run([sys.executable,'-m','pip','uninstall','-y','huggingface_hub'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','huggingface_hub==0.36.2'],check=True)
# Purge any huggingface_hub / transformers cached in this kernel's sys.modules
# so the next import picks up the freshly installed 0.36.2 from disk.
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.') or _mod == 'transformers' or _mod.startswith('transformers.'):
        del sys.modules[_mod]
# Sanity check the import chain that failed before (in this kernel process).
from huggingface_hub.utils import HfFolder  # noqa: F401
from transformers import Seq2SeqTrainer  # noqa: F401
print("IMPORT_OK")

In [ ]:
import tarfile
import torch
assert torch.cuda.is_available(), 'GPU is required; select Kaggle GPU T4.'
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__)
probe = torch.ones((2, 2), device='cuda')
assert (probe @ probe).sum().item() == 8.0
torch.cuda.synchronize(); del probe
print('CUDA_PREFLIGHT_OK', flush=True)
from huggingface_hub import hf_hub_download
from training.protocol import pair_manifest

syn_archive = hf_hub_download('PiotrSty/ocr-pl-lines','ocr-pl-lines-v1.tar.gz',repo_type='dataset',revision=SYN_REVISION)
syn_root = Path('/kaggle/working/ocr-pl-lines-v1'); syn_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(syn_archive,'r:gz') as b: b.extractall(syn_root, filter='data')

ehri_archive = hf_hub_download('PiotrSty/ehri-pl-lines','ehri-pl-lines-v1.tar.gz',repo_type='dataset',revision=EHRI_REVISION)
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1'); ehri_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(ehri_archive,'r:gz') as b: b.extractall(ehri_root, filter='data')

# Augment EHRI train 5× (349 → 2094 lines). Dev/test stay untouched.
ehri_aug_root = Path('/kaggle/working/ehri-pl-lines-aug-v1')
subprocess.run([sys.executable,'-m','training.augment_lines',
    '--input',f'{ehri_root}/train','--output',f'{ehri_aug_root}/train',
    '--copies','5','--seed','42'],check=True)

print('synthetic train:', len(pair_manifest(syn_root/'train')), 'val:', len(pair_manifest(syn_root/'val')))
print('ehri train (orig):', len(pair_manifest(ehri_root/'train')), 'dev:', len(pair_manifest(ehri_root/'dev')), 'test:', len(pair_manifest(ehri_root/'test')))
print('ehri train (aug):', len(pair_manifest(ehri_aug_root/'train')))

In [ ]:
# Fine-tune on synthetic + real EHRI + augmented EHRI; 15 epochs; validate on held-out EHRI dev.
output = '/kaggle/working/trocr-pl-run6'
subprocess.run([sys.executable,'-m','training.train_trocr_pl',
    '--train-dir',f'{syn_root}/train',f'{ehri_root}/train',f'{ehri_aug_root}/train',
    '--val-dir',f'{ehri_root}/dev',
    '--base',BASE_MODEL,'--revision',BASE_REVISION,'--output',output,
    '--epochs','15','--batch-size','8','--lr','1e-4','--no-4bit'],check=True)
print(Path(output,'selection.json').read_text())
print(Path(output,'best_metrics.json').read_text())
# No upload_folder: review CER on held-out test before any promotion.

In [ ]:
# Final evaluation on frozen held-out sets — self-contained cell (safe to rerun alone).
# Compares run6 vs run5 (mixed-v3) vs run4 (mixed-v2) vs run2-base on EHRI test and real-lines-v1.
import sys, subprocess
from pathlib import Path
sys.path.insert(0, '/kaggle/working/OCR_engine')
BASE_MODEL = 'PiotrSty/trocr-pl-base'
RUN4_MODEL = 'PiotrSty/trocr-pl-mixed-v2'
RUN5_MODEL = 'PiotrSty/trocr-pl-mixed-v3'
output = '/kaggle/working/trocr-pl-run6'
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')
real_lines = '/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'
for name, model in [('run6', output), ('run5', RUN5_MODEL), ('run4', RUN4_MODEL), ('run2-base', BASE_MODEL)]:
    for split, data in [('ehri-test', f'{ehri_root}/test'), ('real-lines-v1', real_lines)]:
        print(f'=== {name} on {split} ===', flush=True)
        subprocess.run([sys.executable,'-m','training.evaluate','--data',data,
            '--model',model,'--device','cuda','--batch-size','16'], check=True)

In [ ]:
# Publish run6 to Hugging Face as a SEPARATE experimental model.
# Does NOT overwrite previous models. Uses Kaggle Secret "HF_TOKEN".
import os, json
from pathlib import Path
from huggingface_hub import HfApi, create_repo

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
assert token, "Add a Kaggle Secret named HF_TOKEN with write scope."

api = HfApi(token=token)
repo_id = "PiotrSty/trocr-pl-mixed-aug-v1"
create_repo(repo_id, repo_type="model", exist_ok=True, token=token)

model_dir = "/kaggle/working/trocr-pl-run6"
assert Path(model_dir, "model.safetensors").exists(), f"missing merged model in {model_dir}"

sel = json.loads(Path(model_dir, 'selection.json').read_text())
met = json.loads(Path(model_dir, 'best_metrics.json').read_text())
best_ckpt = sel.get('best_checkpoint', '?')
best_cer = met.get('eval_cer', float('nan'))
best_wer = met.get('eval_wer', float('nan'))

card = f'''---
language: [pl]
license: apache-2.0
base_model: PiotrSty/trocr-pl-base
tags: [trocr, ocr, polish, historical, typewriter, qlora, peft, mixed-data, augmented]
library_name: transformers
---

# PiotrSty/trocr-pl-mixed-aug-v1 (experimental)

Fine-tune of **PiotrSty/trocr-pl-base** on synthetic Polish print + real EHRI
typewritten Polish lines (CC-BY 4.0, ehri-pl-lines) + **5× augmented** EHRI
train lines. run 5 plateaued at 30 epochs on 349 real lines, so this run uses
augmentations (blur, noise, contrast, gamma, JPEG, rotation) to multiply the
real typewriter data 5× (349 → 2094 lines) without new annotations.

## Training

- Base: PiotrSty/trocr-pl-base
- Method: QLoRA on decoder attention (q/k/v/out_proj), rank 16, alpha 32
- Train: 2000 synthetic + 349 real EHRI + 1745 augmented EHRI = 4094 lines
- Val: 38 EHRI lines (held-out doc ZIH3010905, not augmented)
- Epochs: 15, batch 8, lr 1e-4, T4 x2
- Best checkpoint: {best_ckpt} (val CER {best_cer:.4f}, WER {best_wer:.4f})
- Document-level split, no line leakage. Augmentations only on train.

## Evaluation on frozen held-out sets

| Model | EHRI test (81, typewriter) | real-lines-v1 (75, print) |
|---|---:|---:|
| trocr-pl-base (run2) | CER 47.30% / WER 90.82% | CER 11.11% / WER 35.84% |
| trocr-pl-mixed-v2 (run4, 15 ep) | CER 30.42% / WER 78.85% | CER 5.47% / WER 23.20% |
| trocr-pl-mixed-v3 (run5, 30 ep) | CER 29.74% / WER 74.96% | CER 5.36% / WER 22.88% |
| **trocr-pl-mixed-aug-v1 (run6, 15 ep + aug)** | **CER 27.30% / WER 69.83%** | **CER 7.37% / WER 26.72%** |

Augmentations improved typewriter CER by 8% relative to run5 (29.74% → 27.30%)
and WER by 7% (74.96% → 69.83%). However, print CER regressed from 5.36% to
7.37% (+37% relative) — the first domain trade-off observed in this series.
Augmentations tuned for typewriter degradation introduced noise that slightly
hurts high-contrast print recognition. Use run5 (mixed-v3) for print-heavy
workloads; use run6 (mixed-aug-v1) for typewriter-heavy workloads.

## Limitations

- Line recognizer only; page segmentation on faded typewriter is unreliable.
- Augmentations are synthetic; real data diversity still matters.
- Print regression vs run5 — first trade-off in the series.
- Do NOT use as drop-in replacement without your own eval.

## Provenance

See run.json, selection.json, best_metrics.json in this repo.
Source: https://github.com/PiotrStyla/OCR_engine (commit 88a4f7e)
EHRI dataset: https://huggingface.co/datasets/PiotrSty/ehri-pl-lines
'''
Path(model_dir, "README.md").write_text(card, encoding="utf-8")

api.upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
    token=token,
    ignore_patterns=["checkpoint-*", "optimizer.pt", "scheduler.pt", "rng_state*.pth", "training_args.bin"],
)
print(f"Published: https://huggingface.co/{repo_id}")